# YOLOv9 Fine-tune — Michelin Challenge

Pipeline:
1. Instalar dependencias
2. Dividir el dataset en train/val
3. Fine-tune YOLOv9 desde pesos COCO
4. Evaluar el modelo
5. Inferencia visual

**Clases (orden Roboflow):**
- `0: cut_edge` — borde de corte
- `1: qr_code` — código QR
- `2: rubber_strip` — banda de goma completa

**Estructura esperada antes de ejecutar:**
```
data/
├── data.yaml
├── train/
│   ├── images/   ← 22 imágenes de Roboflow
│   └── labels/   ← 22 .txt de Roboflow
└── raw/          ← originales (no se tocan)
```

## 1. Instalación

In [1]:
# %pip install ultralytics --quiet

import ultralytics
ultralytics.checks()

Ultralytics 8.4.48  Python-3.11.14 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Setup complete  (16 CPUs, 15.7 GB RAM, 361.8/476.7 GB disk)


## 2. Split train / val

Mueve el 20% de las imágenes de `train/` a `valid/` para validación.

In [1]:
import os, shutil, random
from pathlib import Path

DATA_ROOT   = Path("../data/new_samples")
TRAIN_IMG   = DATA_ROOT / "train" / "images"
TRAIN_LBL   = DATA_ROOT / "train" / "labels"
VALID_IMG   = DATA_ROOT / "valid" / "images"
VALID_LBL   = DATA_ROOT / "valid" / "labels"
VALID_RATIO = 0.2
SEED        = 42

VALID_IMG.mkdir(parents=True, exist_ok=True)
VALID_LBL.mkdir(parents=True, exist_ok=True)

all_imgs = sorted(TRAIN_IMG.glob("*.jpg")) + sorted(TRAIN_IMG.glob("*.png"))
print(f"Imagenes en train: {len(all_imgs)}")

random.seed(SEED)
random.shuffle(all_imgs)
n_val = max(1, int(len(all_imgs) * VALID_RATIO))
val_imgs = all_imgs[:n_val]

moved = 0
for img in val_imgs:
    lbl = TRAIN_LBL / (img.stem + ".txt")
    shutil.move(str(img), VALID_IMG / img.name)
    if lbl.exists():
        shutil.move(str(lbl), VALID_LBL / lbl.name)
    moved += 1

print(f"Movidas a valid: {moved}")
print(f"Quedan en train: {len(all_imgs) - moved}")

Imagenes en train: 22
Movidas a valid: 4
Quedan en train: 18


## 3. Fine-tune YOLOv9

In [2]:
from ultralytics import YOLO

DATA_YAML = "../data/new_samples/data.yaml"
MODEL     = "yolov9c.pt"
EPOCHS    = 100
IMG_SIZE  = 640
BATCH     = 8
PROJECT   = "../data/runs"
RUN_NAME  = "michelin_v2"

model = YOLO(MODEL)

results = model.train(
    data      = DATA_YAML,
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH,
    project   = PROJECT,
    name      = RUN_NAME,
    device    = 0,
    patience  = 20,
    augment   = True,
    degrees   = 15,
    translate = 0.1,
    scale     = 0.3,
    fliplr    = 0.5,
    mosaic    = 0.5,
)

Ultralytics 8.4.48  Python-3.11.14 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/new_samples/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov9c.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=michelin_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

## 4. Evaluación

In [5]:
from ultralytics import YOLO

BEST_WEIGHTS = "runs/data/runs/michelin_v2/weights/best.pt"
model = YOLO(BEST_WEIGHTS)

metrics = model.val(data="../data/new_samples/data.yaml", imgsz=640)
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

Ultralytics 8.4.48  Python-3.11.14 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLOv9c summary (fused): 156 layers, 25,321,561 parameters, 0 gradients, 102.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1841.5843.9 MB/s, size: 286.7 KB)
val: Scanning C:\Users\jaime\Proyectos\VA_RetoMichelin\models\Jaime\data\new_samples\valid\labels.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.1s/it 3.1s
                   all          4         32      0.773      0.661       0.74      0.516
              cut_edge          3         12      0.381       0.25      0.252     0.0698
               qr_code          4         12          1      0.733      0.971      0.739
          rubber_strip          4          8      0.937          1      0.995      0.739
Speed: 1.9ms preprocess, 99.6ms inference, 0.0ms loss, 6.7ms postprocess per ima

## 5. Inferencia visual

In [6]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from ultralytics import YOLO

BEST_WEIGHTS = "runs/data/runs/michelin_v2/weights/best.pt"
model = YOLO(BEST_WEIGHTS)

CLASS_COLORS = {0: (0, 200, 255), 1: (255, 220, 0), 2: (255, 140, 0)}
CLASS_NAMES  = {0: "cut_edge", 1: "qr_code", 2: "rubber_strip"}

test_images = sorted(Path("../data/raw").glob("*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, test_images):
    img_bgr = cv2.imread(str(img_path))
    preds   = model(img_bgr, verbose=False)[0]

    overlay = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    for box in preds.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        col  = CLASS_COLORS.get(cls, (255, 255, 255))
        cv2.rectangle(overlay, (x1, y1), (x2, y2), col, 3)
        cv2.putText(overlay, f"{CLASS_NAMES[cls]} {conf:.2f}",
                    (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)

    ax.imshow(overlay)
    ax.set_title(img_path.name)
    ax.axis("off")

plt.tight_layout()
plt.show()

<Figure size 1800x1000 with 6 Axes>